# Whisker-Pole Contact Classifier — Inference

Run the trained EfficientNet-B3 on a video and produce:
1. **Per-frame CSV** — every frame with its contact prediction (0/1) and probability
2. **Interval CSV** — contiguous contact regions in `Start,End` format

Batch
1, 3, 5, 15, 16

In [1]:
import sys, os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

INFERENCE_DIR = os.path.dirname(os.path.abspath("__file__"))
if INFERENCE_DIR not in sys.path:
    sys.path.insert(0, INFERENCE_DIR)

from utils import (
    load_model,
    get_inference_transforms,
    VideoFrameDataset,
    run_inference,
    frames_to_intervals,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.5.0+cu124
CUDA available: True
GPU: NVIDIA L40S


## 1 — Configuration

In [2]:
# ========================  PATHS  ========================

VIDEO_PATH = "/home/mvdokh/data/052725_1/052725_piezo_1.mp4"


CHECKPOINT_PATH = os.path.join(
    os.path.dirname(INFERENCE_DIR),
    "Training", "checkpoints", "contact_classifier.pt"
)

# Output CSVs will be saved here
OUTPUT_DIR = "/home/mvdokh/data/052725_1/"
#OUTPUT_DIR = os.path.join(INFERENCE_DIR, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================  SETTINGS  ========================

IMG_SIZE      = 256
BATCH_SIZE    = 64       # can be larger than training since no gradients
NUM_WORKERS   = 2        # try higher to parallelize video decoding
THRESHOLD     = 0.5      # probability threshold for contact
START_FRAME   = 0
END_FRAME     = 250_000  # only first 250k frames (second half has no pole)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

Device: cuda
Checkpoint: /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Training/checkpoints/contact_classifier.pt
Output dir: /home/mvdokh/data/052725_1/


## 2 — Load Model

In [3]:
model = load_model(CHECKPOINT_PATH, DEVICE)

Loaded checkpoint from: /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Training/checkpoints/contact_classifier.pt
  Config: {'img_size': 256, 'batch_size': 32, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'freeze_backbone': False, 'model_arch': 'efficientnet_b3'}
  Epoch: 13
  Val loss: 0.0013


## 3 — Create Video Dataset & DataLoader

In [4]:
transform = get_inference_transforms(IMG_SIZE)

dataset = VideoFrameDataset(
    video_path=VIDEO_PATH,
    start_frame=START_FRAME,
    end_frame=END_FRAME,
    transform=transform,
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Total frames to process: {len(dataset):,}")
print(f"Batches: {len(dataloader):,}")

VideoFrameDataset: frames 0–249999 (250,000 frames)
Total frames to process: 250,000
Batches: 3,907


## 4 — Run Inference

In [5]:
results_df = run_inference(model, dataloader, DEVICE, threshold=THRESHOLD)

n_contact = (results_df["contact"] == 1).sum()
n_no_contact = (results_df["contact"] == 0).sum()
print(f"\nTotal frames   : {len(results_df):,}")
print(f"Contact frames : {n_contact:,} ({100*n_contact/len(results_df):.1f}%)")
print(f"No contact     : {n_no_contact:,} ({100*n_no_contact/len(results_df):.1f}%)")
print()
results_df.head(10)

Inference:   0%|          | 0/3907 [00:00<?, ?it/s]

Inference: 100%|██████████| 3907/3907 [13:22<00:00,  4.87it/s]



Total frames   : 250,000
Contact frames : 75,570 (30.2%)
No contact     : 174,430 (69.8%)



,frame,probability,contact
0,0,0.012066,0
1,1,0.010965,0
2,2,0.014410,0
3,3,0.006684,0
4,4,0.014308,0
5,5,0.013506,0
6,6,0.024055,0
7,7,0.029787,0
8,8,0.043652,0
9,9,0.033921,0


## 5 — Convert to Intervals & Save CSVs

In [6]:
# Per-frame CSV
per_frame_path = os.path.join(OUTPUT_DIR, "per_frame_predictions.csv")
results_df.to_csv(per_frame_path, index=False)
print(f"Saved per-frame predictions → {per_frame_path}")

# Contact intervals CSV
intervals_df = frames_to_intervals(results_df, label_col="contact", label_val=1)
intervals_path = os.path.join(OUTPUT_DIR, "contact_intervals.csv")
intervals_df.to_csv(intervals_path, index=False)
print(f"Saved contact intervals     → {intervals_path}")
print(f"\nFound {len(intervals_df)} contact intervals")
print()
intervals_df.head(20)

Saved per-frame predictions → /home/mvdokh/data/052725_1/per_frame_predictions.csv
Saved contact intervals     → /home/mvdokh/data/052725_1/contact_intervals.csv

Found 2744 contact intervals



,Start,End
0,7329,7347
1,7349,7349
2,9760,9762
3,10242,10242
4,10245,10247
5,10251,10251
6,10266,10279
7,10889,10892
8,10895,10895
9,10898,10901


## 6 — Quick Summary

In [7]:
# Interval length stats
intervals_df["length"] = intervals_df["End"] - intervals_df["Start"] + 1

print(f"Contact intervals: {len(intervals_df)}")
print(f"Total contact frames: {intervals_df['length'].sum():,}")
print(f"Avg interval length: {intervals_df['length'].mean():.1f} frames")
print(f"Min interval length: {intervals_df['length'].min()} frames")
print(f"Max interval length: {intervals_df['length'].max()} frames")
print(f"Median interval length: {intervals_df['length'].median():.0f} frames")

Contact intervals: 2744
Total contact frames: 75,570
Avg interval length: 27.5 frames
Min interval length: 1 frames
Max interval length: 3070 frames
Median interval length: 12 frames
